# Multi-Factor Timing (MT): paper replication

This notebook builds and walk-forward trains the core model from Cotturo, Liu, and Proner
(2025), "Multi-Factor Timing with Deep Learning" — a multi-task neural network that forecasts
the sign of next-month returns for five Fama-French/momentum factors (SMB, HML, RMW, CMA, MOM)
from a shared set of macro and financial predictors. It's the first notebook to run: the other
two (`off-the-shelf-models.ipynb`, `mc-dropout-extension.ipynb`) read data through the same
`src/loading.py` / `src/estimation.py` helpers and compare against MT's results here.

**Architecture**: 4 shared "hard-sharing" dense layers (32 units, batch norm + ReLU) feed into
5 factor-specific heads (2 dense layers each), so the factors share a common representation but
each gets its own final decision boundary — the paper's structure for exploiting cross-factor
information without forcing all five to move together.

**What this notebook does:**
1. Load the response factors, macro predictors, and financial predictors into one panel via the
   shared `est`/`loading` helpers, and check the resulting shapes and class balance.
2. Build the MT architecture and sanity-check it trains on fake data before committing to a
   32-year loop.
3. Sanity-check the walk-forward fold boundaries (expanding training window, 2-year validation
   window immediately before the test year, single held-out test year — paper Sec 3.2.4).
4. Run the full walk-forward loop (1990-2021): retrain from scratch each year, predict the held-out
   year, and save out-of-sample predictions to `results/mt_oos_predictions.csv`.
5. Benchmark OOS classification accuracy (paper Table 1) and multi-factor timing Sharpe ratio
   (paper Table 3) against the paper's published MT numbers.

**Simplification vs. the paper**: one seed per fold with fixed hyperparameters (l1=0.01,
lr=0.001 — both valid points in the paper's Table IA1 grid), rather than the paper's full
hyperparameter grid search + 10-seed ensemble per fold. This is the main source of the gap
between this notebook's numbers and the paper's published ones — see the README for the exact
gap and run-to-run noise.

In [1]:
import sys, os
sys.path.append(os.path.abspath('../src'))

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping

import loading
import estimation as est

In [2]:
# Same shared helper the off-the-shelf and MC Dropout notebooks use, so the OOS split,
# standardization, and scoring logic (paper Sections 3.1, 3.2.4) stay identical across all three.
# Loaded before the model-building sanity check below so n_features can be read off the real
# panel instead of hardcoded.
data, feature_cols = est.build_labels_and_panel(
    loading.response_factors, loading.macro_predictors, loading.financial_predictors
)

print("data shape:", data.shape)
print("features:", len(feature_cols))
print("date range:", data.index.min().date(), "to", data.index.max().date())
print("NaNs in features:", data[feature_cols].isna().sum().sum())
print("label columns:", [c for c in data.columns if c.endswith('_label')])
print("\nclass balance (share positive):")
print(data[[c for c in data.columns if c.endswith('_label')]].mean().round(3))

data shape: (683, 264)
features: 259
date range: 1965-01-31 to 2021-11-30
NaNs in features: 4
label columns: ['SMB_label', 'HML_label', 'RMW_label', 'CMA_label', 'MOM_label']

class balance (share positive):
SMB_label    0.518
HML_label    0.534
RMW_label    0.558
CMA_label    0.526
MOM_label    0.618
dtype: float64


In [3]:
FACTOR_NAMES = est.FACTOR_NAMES

def build_mt_model(n_features, l1_value=0.01, learning_rate=0.001, seed=0):
    tf.random.set_seed(seed)
    np.random.seed(seed)

    inputs = Input(shape=(n_features,), name='predictors')
    x = inputs

    # 4 shared "hard-sharing" layers — 32 units, batch norm + ReLU after each
    for i in range(4):
        x = layers.Dense(32, kernel_regularizer=regularizers.l1(l1_value),
                          name=f'shared_dense_{i+1}')(x)
        x = layers.BatchNormalization(name=f'shared_bn_{i+1}')(x)
        x = layers.ReLU(name=f'shared_relu_{i+1}')(x)

    shared_latent = x

    outputs = []
    for factor in FACTOR_NAMES:
        f = layers.Dense(8, activation='relu', kernel_regularizer=regularizers.l1(l1_value),
                          name=f'{factor}_dense1')(shared_latent)
        f = layers.Dense(8, activation='relu', kernel_regularizer=regularizers.l1(l1_value),
                          name=f'{factor}_dense2')(f)
        f_out = layers.Dense(1, activation='sigmoid', name=f'{factor}_output')(f)
        outputs.append(f_out)

    model = Model(inputs=inputs, outputs=outputs, name='MT')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss={f'{factor}_output': 'binary_crossentropy' for factor in FACTOR_NAMES},
    )
    return model

# --- Sanity check with fake data ---
n_features = len(feature_cols)
model = build_mt_model(n_features)
model.summary()

# fake batch: 20 fake "months", random predictors, random binary labels
X_fake = np.random.randn(20, n_features).astype('float32')
y_fake = {f'{f}_output': np.random.randint(0, 2, 20).astype('float32') for f in FACTOR_NAMES}

history = model.fit(X_fake, y_fake, epochs=2, batch_size=4, verbose=1)
print("\n Model builds and trains without error.")

Model: "MT"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ predictors          │ (None, 259)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense_1      │ (None, 32)        │      8,320 │ predictors[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bn_1         │ (None, 32)        │        128 │ shared_dense_1[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_relu_1       │ (None, 32)        │          0 │ shared_bn_1[0][0] │
│ (ReLU)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense_2      │ (None, 32)        │      1,056 │ shared_relu_1[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bn_2         │ (None, 32)        │        128 │ shared_dense_2[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_relu_2       │ (None, 32)        │          0 │ shared_bn_2[0][0] │
│ (ReLU)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense_3      │ (None, 32)        │      1,056 │ shared_relu_2[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bn_3         │ (None, 32)        │        128 │ shared_dense_3[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_relu_3       │ (None, 32)        │          0 │ shared_bn_3[0][0] │
│ (ReLU)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense_4      │ (None, 32)        │      1,056 │ shared_relu_3[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bn_4         │ (None, 32)        │        128 │ shared_dense_4[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_relu_4       │ (None, 32)        │          0 │ shared_bn_4[0][0] │
│ (ReLU)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ SMB_dense1 (Dense)  │ (None, 8)         │        264 │ shared_relu_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ HML_dense1 (Dense)  │ (None, 8)         │        264 │ shared_relu_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ RMW_dense1 (Dense)  │ (None, 8)         │        264 │ shared_relu_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ CMA_dense1 (Dense)  │ (None, 8)         │        264 │ shared_relu_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ MOM_dense1 (Dense)  │ (None, 8)         │        264 │ shared_relu_4[0]… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 13,725 (53.61 KB)

 Trainable params: 13,469 (52.61 KB)

 Non-trainable params: 256 (1.00 KB)

...

5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - CMA_output_loss: 0.7253 - HML_output_loss: 0.7062 - MOM_output_loss: 0.7641 - RMW_output_loss: 0.6968 - SMB_output_loss: 0.6760 - loss: 17.7315


Epoch 2/2


1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - CMA_output_loss: 0.6274 - HML_output_loss: 0.7639 - MOM_output_loss: 0.7315 - RMW_output_loss: 0.6940 - SMB_output_loss: 0.6263 - loss: 17.5468

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - CMA_output_loss: 0.6650 - HML_output_loss: 0.6694 - MOM_output_loss: 0.7158 - RMW_output_loss: 0.6791 - SMB_output_loss: 0.6435 - loss: 17.4394



 Model builds and trains without error.


In [4]:
# Walk-forward estimation procedure (paper Sec 3.2.4): train on an expanding window, validate
# on the 2 years immediately before the test year (for early stopping), test on 1 held-out year.
# Sanity-check the fold boundaries and resulting shapes before committing to the full 32-year loop.
print("first OOS fold (test 1990):", est.fold_years(1990))
print("last OOS fold  (test 2021):", est.fold_years(2021))
print("total OOS test years:", len(list(est.OOS_TEST_YEARS)))

train, val, test, X_train, X_val, X_test = est.prepare_fold(data, feature_cols, 1990)
print("\n1990 fold -> train:", X_train.shape, " val:", X_val.shape, " test:", X_test.shape)
train, val, test, X_train, X_val, X_test = est.prepare_fold(data, feature_cols, 2021)
print("2021 fold -> train:", X_train.shape, " val:", X_val.shape, " test:", X_test.shape)

first OOS fold (test 1990): ('1987-12-31', '1988-01-01', '1989-12-31', '1990-01-01', '1990-12-31')
last OOS fold  (test 2021): ('2018-12-31', '2019-01-01', '2020-12-31', '2021-01-01', '2021-12-31')
total OOS test years: 32

1990 fold -> train: (276, 259)  val: (24, 259)  test: (12, 259)
2021 fold -> train: (648, 259)  val: (24, 259)  test: (11, 259)


In [5]:
# Walk-forward re-estimation matching the paper's estimation procedure (Sec 3.2.4): retrain MT
# from scratch each year on an expanding window, using the 2 years right before the test year
# as an early-stopping validation set, then predict the single held-out test year.
#
# Simplification vs. the paper: this trains one seed per fold with the notebook's default
# hyperparameters (l1=0.01, lr=0.001 — both valid points in the paper's Table IA1 grid) rather
# than the paper's full hyperparameter grid search + 10-seed ensemble per fold, which would take
# far longer to run here. Grid search / ensembling would be the natural next step to close the
# gap with the paper's published numbers.

oos_records = []

for test_year in est.OOS_TEST_YEARS:
    train, val, test, X_train, X_val, X_test = est.prepare_fold(data, feature_cols, test_year)

    y_train = {f'{f}_output': train[f'{f}_label'].values.astype('float32') for f in FACTOR_NAMES}
    y_val = {f'{f}_output': val[f'{f}_label'].values.astype('float32') for f in FACTOR_NAMES}

    model = build_mt_model(n_features=X_train.shape[1])
    es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    history = model.fit(
        X_train.values.astype('float32'), y_train,
        validation_data=(X_val.values.astype('float32'), y_val),
        epochs=200, batch_size=4, callbacks=[es], verbose=0,
    )

    preds = model.predict(X_test.values.astype('float32'), verbose=0)
    fold_df = pd.DataFrame(
        {f'{f}_prob': np.asarray(preds[i]).flatten() for i, f in enumerate(FACTOR_NAMES)},
        index=test.index,
    )
    oos_records.append(fold_df)
    print(f'{test_year}: train={X_train.shape[0]}mo  val={X_val.shape[0]}mo'
          f'epochs_run={len(history.history["loss"])}')

oos_predictions = pd.concat(oos_records).sort_index()
oos_predictions.to_csv('../results/mt_oos_predictions.csv')
print("\nOOS predictions:", oos_predictions.shape)
oos_predictions.head()

1990: train=276mo  val=24moepochs_run=58


1991: train=288mo  val=24moepochs_run=66


1992: train=300mo  val=24moepochs_run=62


1993: train=312mo  val=24moepochs_run=62


1994: train=324mo  val=24moepochs_run=85


...

2018: train=612mo  val=24moepochs_run=113


2019: train=624mo  val=24moepochs_run=110


2020: train=636mo  val=24moepochs_run=56


2021: train=648mo  val=24moepochs_run=52

OOS predictions: (383, 5)


,SMB_prob,HML_prob,RMW_prob,CMA_prob,MOM_prob
1990-01-31,0.443381,0.921758,0.226500,0.882895,0.457163
1990-02-28,0.161622,0.762724,0.830495,0.843117,0.850435
1990-03-31,0.401044,0.953626,0.261096,0.915869,0.428605
1990-04-30,0.918452,0.248486,0.420174,0.141582,0.627719
1990-05-31,0.887832,0.314114,0.467476,0.223610,0.757468


In [6]:
# Benchmark: out-of-sample classification accuracy (paper Table 1) and multi-factor timing
# performance (paper Table 3) for MT, evaluated against the naive buy-and-hold (BUY) benchmark,
# with the paper's published MT/BUY numbers alongside for reference.

buy_acc = est.buy_and_hold_accuracy(data, oos_predictions.index)
mt_acc = est.classification_accuracy(oos_predictions, data)
paper_mt_acc = pd.Series({'SMB': 53.6, 'HML': 55.5, 'RMW': 57.6, 'CMA': 51.0, 'MOM': 59.4, 'Mean': 55.4})
paper_buy_acc = pd.Series({'SMB': 51.6, 'HML': 47.9, 'RMW': 58.3, 'CMA': 49.2, 'MOM': 60.4, 'Mean': 53.5})

accuracy_table = pd.DataFrame({
    'BUY (this notebook)': (buy_acc * 100).round(1),
    'BUY (paper)': paper_buy_acc,
    'MT (this notebook)': (mt_acc * 100).round(1),
    'MT (paper)': paper_mt_acc,
})
print("Out-of-sample classification accuracy, % (paper Table 1)")
print(accuracy_table)

mt_strategy = est.strategy_returns(oos_predictions, loading.response_factors)
# same t -> t+1 alignment as est.strategy_returns (see its docstring): a signal formed at t is
# graded against t+1's realized return, so the BUY benchmark must be shifted the same way or the
# spanning regression below pairs mismatched months.
buy_ew = loading.response_factors.rename(columns={'Mom': 'MOM'}).shift(-1).loc[oos_predictions.index, FACTOR_NAMES].mean(axis=1)

mt_sharpe = est.annualized_sharpe(mt_strategy['EW'])
buy_sharpe = est.annualized_sharpe(buy_ew)
span = est.spanning_regression(mt_strategy['EW'], buy_ew)

perf_table = pd.DataFrame({
    'this notebook': pd.concat([pd.Series({'Sharpe Ratio': mt_sharpe}), span]),
    'paper (MT)': pd.Series({'Sharpe Ratio': 0.69, 'alpha (annualized %)': 0.70, 't(alpha)': 1.91,
                              'beta': 0.83, 'R2 (%)': 82.20}),
})
print("\nMulti-factor timing performance vs. multi-factor BUY (paper Table 3)")
print(perf_table.round(3))
print(f"\nmulti-factor BUY Sharpe -- this notebook: {buy_sharpe:.2f}  paper: 0.60")

mt_strategy.to_csv('../results/mt_strategy_returns.csv')

Out-of-sample classification accuracy, % (paper Table 1)
      BUY (this notebook)  BUY (paper)  MT (this notebook)  MT (paper)
SMB                  51.7         51.6                53.5        53.6
HML                  47.8         47.9                51.7        55.5
RMW                  58.5         58.3                55.6        57.6
CMA                  49.6         49.2                52.0        51.0
MOM                  60.1         60.4                58.7        59.4
Mean                 53.5         53.5                54.3        55.4

Multi-factor timing performance vs. multi-factor BUY (paper Table 3)
                      this notebook  paper (MT)
Sharpe Ratio                  0.666        0.69
alpha (annualized %)          0.680        0.70
t(alpha)                      1.473        1.91
beta                          0.669        0.83
R2 (%)                       68.991       82.20

multi-factor BUY Sharpe -- this notebook: 0.60  paper: 0.60
